# 01 — LSTM Baseline
This notebook trains a baseline Long Short-Term Memory (LSTM) network to forecast 5-day stock returns.
The model is trained jointly across the dataset and provides deterministic point predictions.

In [1]:
import os, sys, numpy as np, pandas as pd, torch, torch.nn as nn, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from torch.utils.data import TensorDataset, DataLoader
sys.path.insert(0, os.path.abspath('..'))
DATA_PATH='../data'; SEQ_LEN=60; FORECAST=5; NUM_STOCKS=50; TRAIN_R=0.8; TGT=1; BATCH=64
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Ready | device:', device)


Ready | device: cpu


## Data

In [2]:
def make_seq(data):
    X,y=[],[]
    for i in range(len(data)-SEQ_LEN-FORECAST+1):
        X.append(data[i:i+SEQ_LEN])
        y.append(data[i+SEQ_LEN:i+SEQ_LEN+FORECAST, TGT])
    return np.array(X), np.array(y)

def load_data():
    files=[f for f in sorted(os.listdir(DATA_PATH)) if f.endswith('.txt')][:NUM_STOCKS]
    dfs=[]
    for f in files:
        try:
            df=pd.read_csv(os.path.join(DATA_PATH,f))
            if not {'Date','Close'}.issubset(df.columns): continue
            df['Date']=pd.to_datetime(df['Date']); df=df.sort_values('Date')
            df['Return']=df['Close'].pct_change(); df['MA_10']=df['Close'].rolling(10).mean()
            df['MA_50']=df['Close'].rolling(50).mean(); df=df.dropna(); df['Stock']=f
            if len(df)>SEQ_LEN+FORECAST: dfs.append(df)
        except: pass
    return pd.concat(dfs,ignore_index=True)

cdf=load_data()
all_X,all_y=[],[]
for s in cdf['Stock'].unique():
    sdf=cdf[cdf['Stock']==s]; sc=MinMaxScaler()
    d=sc.fit_transform(sdf[['Close','Return','MA_10','MA_50']]); Xs,ys=make_seq(d)
    if len(Xs): all_X.append(Xs); all_y.append(ys)
X=np.concatenate(all_X); y=np.concatenate(all_y)
sp=int(TRAIN_R*len(X)); X_tr,X_te=X[:sp],X[sp:]; y_tr,y_te=y[:sp],y[sp:]
T=lambda a: torch.tensor(a,dtype=torch.float32).to(device)
Xt,Xe,yt_t,ye_t=T(X_tr),T(X_te),T(y_tr),T(y_te)
print(f'Train:{len(X_tr):,} | Test:{len(X_te):,} | Shape X:{X.shape}')


Train:108,644 | Test:27,161 | Shape X:(135805, 60, 4)


## Model — LSTM

In [3]:
from src.lstm_model import LSTMModel


## Training

In [4]:
MODEL_NAME='LSTM'; EPOCHS=25; LR=1e-3
model=LSTMModel(input_size=4).to(device)
print(f'Params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')
opt=torch.optim.Adam(model.parameters(),lr=LR)
ldr=DataLoader(TensorDataset(Xt,yt_t),batch_size=BATCH,shuffle=True)
losses=[]
for ep in range(EPOCHS):
    model.train(); tot=0
    for xb,yb in ldr:
        p=model(xb); loss=nn.MSELoss()(p,yb)
        opt.zero_grad(); loss.backward(); opt.step(); tot+=loss.item()
    avg=tot/len(ldr); losses.append(avg)
    print(f'Epoch {ep+1:02d}/{EPOCHS} MSE:{avg:.6f}')
torch.save(model.state_dict(),'../models/lstm_model.pth')
model.eval()
with torch.no_grad(): mu=model(Xe).cpu().numpy()


Params: 51,525


Epoch 01/25 MSE:0.006097


Epoch 02/25 MSE:0.003456


Epoch 03/25 MSE:0.003437


Epoch 04/25 MSE:0.003416


...

Epoch 21/25 MSE:0.003343


Epoch 22/25 MSE:0.003339


Epoch 23/25 MSE:0.003337


Epoch 24/25 MSE:0.003338


Epoch 25/25 MSE:0.003338


## Save Results

In [5]:
os.makedirs('../outputs',exist_ok=True); os.makedirs('../models',exist_ok=True)
import pandas as pd
rows=[]
for d in range(FORECAST):
    mae=mean_absolute_error(y_te[:,d],mu[:,d]); rmse=float(np.sqrt(mean_squared_error(y_te[:,d],mu[:,d])))
    rows.append({'model':MODEL_NAME,'day':f'Day+{d+1}','mae':round(mae,6),'rmse':round(rmse,6)})
ov_mae=mean_absolute_error(y_te.flatten(),mu.flatten()); ov_rmse=float(np.sqrt(mean_squared_error(y_te.flatten(),mu.flatten())))
rows.append({'model':MODEL_NAME,'day':'Overall','mae':round(ov_mae,6),'rmse':round(ov_rmse,6)})
rp='../outputs/results.csv'; new=pd.DataFrame(rows)
if os.path.exists(rp):
    ex=pd.read_csv(rp); ex=ex[ex['model']!=MODEL_NAME]; new=pd.concat([ex,new],ignore_index=True)
new.to_csv(rp,index=False); print(f'Saved {MODEL_NAME} → {rp}')
print(new[new['model']==MODEL_NAME].to_string(index=False))


Saved LSTM → ../outputs/results.csv
model     day      mae     rmse  avg_uncertainty
 LSTM   Day+1 0.040271 0.060021              NaN
 LSTM   Day+2 0.039298 0.058978              NaN
 LSTM   Day+3 0.039523 0.059261              NaN
 LSTM   Day+4 0.039674 0.059265              NaN
 LSTM   Day+5 0.039206 0.059008              NaN
 LSTM Overall 0.039595 0.059308              NaN


## Plots

In [6]:
plt.figure(figsize=(8,4)); plt.plot(losses,color='#3498DB',lw=2)
plt.title('LSTM Training Loss'); plt.xlabel('Epoch'); plt.ylabel('MSE')
plt.tight_layout(); plt.savefig('../outputs/lstm_loss.png',dpi=150); plt.close()
n=min(300,len(y_te)); idx=np.arange(n)
plt.figure(figsize=(14,4))
plt.plot(idx,y_te[:n,0],label='Actual',color='#2ECC71',lw=1.2)
plt.plot(idx,mu[:n,0],label='Predicted',color='#E74C3C',lw=1.2,alpha=0.85)
plt.title('LSTM: Actual vs Predicted Return — Day+1'); plt.xlabel('Sample'); plt.ylabel('Scaled Return')
plt.legend(); plt.tight_layout(); plt.savefig('../outputs/lstm_pred.png',dpi=150); plt.close()
print('Plots saved')


Plots saved
